# 2.2 Automated Feature Extraction

Automatically create features from raw data using Featuretools, tsfresh, and LLMs.

## Topics
- Deep Feature Synthesis with Featuretools
- Time-series feature extraction with tsfresh
- Text feature embedding with Transformers
- LLM-based feature suggestion


In [ ]:
# !pip install featuretools tsfresh transformers sentence-transformers


In [ ]:
import pandas as pd
import featuretools as ft
from sklearn.datasets import fetch_openml

# Load a relational-style dataset
df_customers = pd.DataFrame({
    'customer_id': range(1, 6),
    'age': [25, 34, 45, 23, 56],
    'signup_year': [2020, 2019, 2018, 2021, 2017]
})

df_orders = pd.DataFrame({
    'order_id': range(1, 11),
    'customer_id': [1, 1, 2, 3, 3, 3, 4, 5, 5, 5],
    'amount': [150, 200, 500, 75, 300, 420, 90, 1200, 800, 650],
    'product_category': ['A', 'B', 'A', 'C', 'A', 'B', 'C', 'A', 'B', 'C']
})

print(df_customers.head())
print(df_orders.head())


In [ ]:
# Build EntitySet
es = ft.EntitySet(id='retail')
es = es.add_dataframe(dataframe=df_customers, dataframe_name='customers', index='customer_id')
es = es.add_dataframe(dataframe=df_orders, dataframe_name='orders', index='order_id')

# Add relationship
es = es.add_relationship('customers', 'customer_id', 'orders', 'customer_id')

# Run Deep Feature Synthesis
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    max_depth=2
)

print(f'Generated {len(feature_defs)} features')
print(feature_matrix.head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Embed text features using a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

sample_texts = [
    'Customer complained about delayed delivery',
    'Very satisfied with the product quality',
    'Order was damaged during shipping',
    'Excellent customer service experience'
]

embeddings = model.encode(sample_texts)
print(f'Embedding shape: {embeddings.shape}')
print(f'Each text is represented as a {embeddings.shape[1]}-dimensional vector')


In [ ]:
# LLM-based feature suggestion
from openai import OpenAI

client = OpenAI()

def suggest_features(dataset_description: str) -> str:
    """Ask an LLM to suggest features for a given dataset."""
    response = client.chat.completions.create(
        model='gpt-4',
        messages=[{
            'role': 'user',
            'content': f'I have a dataset with: {dataset_description}\nSuggest 10 new engineered features that could improve a churn prediction model. For each feature, provide the name and the pandas code to create it.'
        }]
    )
    return response.choices[0].message.content

# desc = 'columns: customer_id, age, tenure_months, monthly_charge, num_support_tickets, last_login_days_ago'
# print(suggest_features(desc))


## Key Takeaways
- DFS automatically discovers feature combinations across related tables
- Text embeddings convert unstructured text into dense ML-ready features
- LLMs can suggest domain-specific features that manual analysis might miss
